<a href="https://colab.research.google.com/github/MaSBroEarl/Task.Text-generation/blob/main/LoRA%2BLLMasJudge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Импорт библиотек и загрузка модели:

Qwen/Qwen2.5-1.5B-Instruct

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Загрузка модели...")

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print("✅ Модель загружена!")

In [ ]:
# Берём один диалог
df = pd.read_csv('/content/train.csv')
test_dialogue = df.iloc[0]['dialogue']
test_summary = df.iloc[0]['summary']

prompt = f"""<|im_start|>user
Summarize this dialogue concisely:

{test_dialogue}
<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
prediction = response.split("<|im_start|>assistant")[-1].strip()

print("Референс:", test_summary)
print("Предсказание:", prediction)

Импорт библиотек для LoRA

In [ ]:
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
from transformers import BitsAndBytesConfig

print("✅ LoRA библиотеки импортированы")

Подготовка данных для LoRA

In [ ]:
df = pd.read_csv('/content/train.csv')
df_train = df.sample(500, random_state=42)
df_eval = df.sample(100, random_state=43)

def format_example(row):
    return f"""<|im_start|>user
Summarize this dialogue concisely:

{row['dialogue']}
<|im_end|>
<|im_start|>assistant
{row['summary']}
<|im_end|>"""

train_dataset = Dataset.from_dict({"text": [format_example(row) for _, row in df_train.iterrows()]})
eval_dataset = Dataset.from_dict({"text": [format_example(row) for _, row in df_eval.iterrows()]})

print("✅ Данные готовы")
print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

In [ ]:
def tokenize_function(examples):
    result = tokenizer(examples["text"], truncation=True, padding=False, max_length=512)
    result["labels"] = result["input_ids"].copy()  # ВАЖНО: добавляем labels
    return result

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_eval = eval_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
print("✅ Токенизация готова")
print(f"Train: {len(tokenized_train)}, Eval: {len(tokenized_eval)}")

In [ ]:
!pip install torchao==0.16.0 -q

In [ ]:
# Подготовка модели
model = prepare_model_for_kbit_training(model)

# Конфигурация LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Аргументы обучения
training_args = TrainingArguments(
    output_dir="./qwen-lora",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    logging_steps=10,
    eval_strategy="no",
    save_strategy="no",
    learning_rate=2e-4,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
)

print("🚀 Обучение...")
trainer.train()
print("✅ Обучение завершено!")

# Сохраняем модель
model.save_pretrained("./qwen-lora-final")
tokenizer.save_pretrained("./qwen-lora-final")
print("✅ Модель сохранена!")

Делаем инференс на тестовых примерах

In [ ]:
# Берём 20 тестовых примеров
test_df = df.sample(20, random_state=123)

predictions = []
references = []

for _, row in test_df.iterrows():
    prompt = f"""<|im_start|>user
Summarize this dialogue concisely:

{row['dialogue']}
<|im_end|>
<|im_start|>assistant
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    pred = response.split("<|im_start|>assistant")[-1].strip()

    predictions.append(pred)
    references.append(row['summary'])

# Смотрим результаты
for i in range(5):
    print(f"\n--- Пример {i+1} ---")
    print(f"Референс: {references[i]}")
    print(f"Предсказание: {predictions[i]}")
    print("-"*50)

In [ ]:
!pip install rouge-score -q

In [ ]:
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import numpy as np

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
smoothie = SmoothingFunction().method4

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []
bleu_scores = []

for ref, pred in zip(references, predictions):
    scores = scorer.score(ref, pred)
    rouge1_scores.append(scores['rouge1'].fmeasure)
    rouge2_scores.append(scores['rouge2'].fmeasure)
    rougeL_scores.append(scores['rougeL'].fmeasure)

    bleu = sentence_bleu([ref.split()], pred.split(), smoothing_function=smoothie)
    bleu_scores.append(bleu)

print("="*50)
print("МЕТРИКИ (на 20 примерах)")
print("="*50)
print(f"ROUGE-1: {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2: {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L: {np.mean(rougeL_scores):.4f}")
print(f"BLEU:    {np.mean(bleu_scores):.4f}")

LLM как судья
Groq

In [ ]:
!pip install groq -q

In [ ]:
from groq import Groq

# Вставь свой API ключ
client = Groq(api_key="ТОКЕН")

def judge_summary(dialogue, reference, prediction):
    prompt = f"""You are an expert evaluator of text summarization.

Dialogue:
{dialogue}

Reference Summary:
{reference}

Candidate Summary:
{prediction}

Rate the candidate summary on a scale of 1-5 based on:
1. Relevance (does it capture key points?)
2. Conciseness (is it concise?)
3. Fluency (is it readable?)

Also provide a brief justification.

Output format:
Relevance: [score]
Conciseness: [score]
Fluency: [score]
Justification: [text]
"""
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=300,
    )
    return response.choices[0].message.content

# Проверяем на 5 примерах
for i in range(5):
    print(f"\n--- Пример {i+1} ---")
    print(judge_summary(
        test_df.iloc[i]['dialogue'][:500],
        references[i],
        predictions[i]
    ))
    print("="*50)